# Task 1 — Genetic Algorithm for Job-Shop Scheduling

> Implement a genetic algorithm to schedule jobs using available resources, so that the time to finish all tasks is minimal. Test your algorithm using different parameters and analyse the results. The data is in a separate excel file (`GA_task.xlsx`) – each task has the resource (R) and time needed using this resource (T) in a specific order. You cannot change the order of operations within the task.

## 1. Problem interpretation

Inspection of `GA_task.xlsx` (sheet `Arkusz2`, size 100 columns × 13 rows) shows:

- Row 1: job IDs `1 … 50` (each spans 2 columns: `R` and `T`),
- Row 2: repeating headers `R | T | R | T …`,
- Rows 3–13: 11 operations per job, each cell pair = `(resource/machine R ∈ 1..10, processing time T)`.

This is the classical **Job-Shop Scheduling Problem (JSSP)**:

- **50 jobs × 11 operations = 550 operations**, **10 machines** (resources 1–10).
- Operation `k` of job `j` must run on machine `R[j,k]` for `T[j,k]` time units.
- **Precedence:** operations of one job must run in the given order (no reordering inside a job).
- **Capacity:** a machine processes at most one operation at a time (no preemption).
- **Objective:** minimise **makespan** $C_{max}$ = time when the last operation finishes.

Lower bound: `max(total load of bottleneck machine, longest single job)`. Computed below (≈ 2073). A random schedule gives ≈ 2500–3000, so there is real room for optimisation.

In [ ]:
import openpyxl
import numpy as np
import matplotlib.pyplot as plt
import time, random

# --- Load data ---
wb = openpyxl.load_workbook('GA_task.xlsx', data_only=True)
ws = wb['Arkusz2']
N_JOBS = ws.max_column // 2      # 50
N_OPS  = ws.max_row - 2          # 11
print(f'Sheet: {ws.title}, dims={ws.dimensions} -> {N_JOBS} jobs x {N_OPS} ops')

MACH = np.zeros((N_JOBS, N_OPS), dtype=int)   # 0-based machine id
DUR  = np.zeros((N_JOBS, N_OPS), dtype=int)   # processing time
for j in range(N_JOBS):
    for k in range(N_OPS):
        MACH[j, k] = int(ws.cell(row=3+k, column=2*j+1).value) - 1
        DUR[j, k]  = int(ws.cell(row=3+k, column=2*j+2).value)
N_MACH = int(MACH.max()) + 1
print(f'Machines: {N_MACH} (ids 1..{N_MACH}), total operations: {N_JOBS*N_OPS}')
print(f'Example Job 1 (machine, time): {list(zip(MACH[0]+1, DUR[0]))}')
print(f'Example Job 2 (machine, time): {list(zip(MACH[1]+1, DUR[1]))}')

# --- Basic statistics ---
machine_load = np.zeros(N_MACH, dtype=int)
for j in range(N_JOBS):
    for k in range(N_OPS):
        machine_load[MACH[j, k]] += DUR[j, k]
job_length = DUR.sum(axis=1)
lb = max(machine_load.max(), job_length.max())
print(f'\nTotal work: {DUR.sum()}, lower bound of makespan: {lb}')
print('Load per machine:', {m+1: int(v) for m, v in enumerate(machine_load)})
print(f'Job lengths: min={job_length.min()}, mean={job_length.mean():.1f}, max={job_length.max()}')
print(f'Durations: min={DUR.min()}, max={DUR.max()}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(np.arange(1, N_MACH+1), machine_load, color='steelblue')
ax[0].axhline(lb, color='red', ls='--', label=f'lower bound {lb}')
ax[0].set_xlabel('Machine (resource)'); ax[0].set_ylabel('Total load'); ax[0].legend()
ax[0].set_title('Load per machine — machine 7 is the bottleneck')
ax[1].hist(job_length, bins=12, color='seagreen', edgecolor='white')
ax[1].set_xlabel('Total processing time of a job'); ax[1].set_ylabel('Count')
ax[1].set_title('Distribution of job lengths')
plt.tight_layout(); plt.show()

## 2. Representation and makespan decoder

**Chromosome (genotype):** operation-based permutation with repetitions (Bierwirth representation) — a permutation of length `50 × 11 = 550` in which each job ID `0…49` appears exactly **11** times. The `t`-th occurrence of job `j` means *“schedule operation `t` of job `j` next”*.

**Decoder (phenotype):** left-to-right list scheduler:

```
machine_available[m] = 0 for all machines
job_available[j] = 0, next_op[j] = 0 for all jobs
for gene g in chromosome:
    o = next_op[g];  m = MACH[g,o];  d = DUR[g,o]
    start = max(machine_available[m], job_available[g])   # respects machine capacity + job precedence
    end = start + d
    machine_available[m] = end;  job_available[g] = end;  next_op[g] += 1
makespan = max(machine_available)
```

Every chromosome decodes to a **feasible semi-active schedule**; every (semi-active) schedule can be represented. Fitness = makespan (minimisation). Decoding is `O(550)` and takes ~0.4 ms, so tens of thousands of evaluations are cheap.

**Genetic operators** (all preserve the “11 copies per job” invariant):

- *Initialisation:* shuffled copies of `[0×11, 1×11, …, 49×11]`.
- *Crossover — Job Order Crossover (JOX):* pick a random subset of jobs; child 1 inherits genes of those jobs at the same positions from parent 1 and the remaining genes in the order of parent 2 (and vice-versa for child 2).
- *Mutation — swap:* exchange two random positions (optionally repeated). Small perturbation, always valid.
- *Selection:* tournament (size `k`) + elitism (copy `e` best unchanged).

In [ ]:
def makespan_of(chrom):
    """Makespan of a chromosome (array-like of job ids, each appearing N_OPS times)."""
    mach_av = np.zeros(N_MACH, dtype=np.int32)
    job_av  = np.zeros(N_JOBS, dtype=np.int32)
    nxt     = np.zeros(N_JOBS, dtype=np.int32)
    for g in chrom:
        o = nxt[g]
        m = MACH[g, o]; d = DUR[g, o]
        s = mach_av[m] if mach_av[m] > job_av[g] else job_av[g]
        e = s + d
        mach_av[m] = e; job_av[g] = e
        nxt[g] = o + 1
    return int(mach_av.max())

def decode_schedule(chrom):
    """Full decode returning list of (job, op, machine, start, end) + makespan."""
    mach_av = np.zeros(N_MACH, dtype=int)
    job_av  = np.zeros(N_JOBS, dtype=int)
    nxt     = np.zeros(N_JOBS, dtype=int)
    rec = []
    for g in chrom:
        g = int(g)
        o = nxt[g]; m = int(MACH[g, o]); d = int(DUR[g, o])
        s = max(mach_av[m], job_av[g]); e = s + d
        mach_av[m] = e; job_av[g] = e; nxt[g] += 1
        rec.append((g, o, m, s, e))
    return rec, int(mach_av.max())

def random_chromosome(rng):
    c = np.repeat(np.arange(N_JOBS), N_OPS)
    rng.shuffle(c)
    return c

# --- Sanity checks + baselines ---
rng = np.random.default_rng(0)
base = np.repeat(np.arange(N_JOBS), N_OPS)
c_fifo = base.copy()
print('FIFO (job-ordered) makespan:', makespan_of(c_fifo))
rand_vals = [makespan_of(random_chromosome(rng)) for _ in range(30)]
print(f'Random schedules over 30 trials: min={min(rand_vals)}, mean={np.mean(rand_vals):.0f}, max={max(rand_vals)}')
# validity: each job appears exactly N_OPS times
c = random_chromosome(rng)
assert len(c) == N_JOBS*N_OPS and all((c == j).sum() == N_OPS for j in range(N_JOBS))
print('Chromosome validity OK, example makespan:', makespan_of(c))

## 3. Genetic algorithm

Standard generational GA with tournament selection, JOX crossover, swap mutation and elitism. Fitness = makespan (lower is better).

In [ ]:
def jox_crossover(p1, p2, rng):
    """Job Order Crossover: returns two children, preserves repetition counts."""
    chosen = rng.choice(N_JOBS, size=N_JOBS // 2, replace=False)
    inset = np.zeros(N_JOBS, dtype=bool); inset[chosen] = True
    c1 = np.empty_like(p1); c2 = np.empty_like(p2)
    m1 = inset[p1]; c1[m1] = p1[m1]; c1[~m1] = p2[~inset[p2]]
    m2 = inset[p2]; c2[m2] = p2[m2]; c2[~m2] = p1[~inset[p1]]
    return c1, c2

def swap_mutation(c, rng, n_swaps=1):
    for _ in range(n_swaps):
        i, j = rng.integers(0, len(c), size=2)
        c[i], c[j] = c[j], c[i]
    return c

def run_ga(pop_size=80, n_gen=100, cx_rate=0.85, mut_rate=0.25,
           tour_size=3, n_elite=2, seed=0, verbose=False):
    rng = np.random.default_rng(seed)
    pop = np.array([random_chromosome(rng) for _ in range(pop_size)])
    fit = np.array([makespan_of(c) for c in pop])
    best_hist, mean_hist = [], []
    best_chrom = pop[np.argmin(fit)].copy(); best_fit = int(fit.min())
    for g in range(n_gen):
        order = np.argsort(fit); pop, fit = pop[order], fit[order]
        if int(fit[0]) < best_fit:
            best_fit, best_chrom = int(fit[0]), pop[0].copy()
        best_hist.append(best_fit); mean_hist.append(float(fit.mean()))
        nxt_pop = [p.copy() for p in pop[:n_elite]]
        while len(nxt_pop) < pop_size:
            ia = rng.choice(pop_size, size=tour_size, replace=False)
            ib = rng.choice(pop_size, size=tour_size, replace=False)
            p1, p2 = pop[ia[np.argmin(fit[ia])]], pop[ib[np.argmin(fit[ib])]]
            if rng.random() < cx_rate:
                ch1, ch2 = jox_crossover(p1, p2, rng)
            else:
                ch1, ch2 = p1.copy(), p2.copy()
            if rng.random() < mut_rate: swap_mutation(ch1, rng)
            if rng.random() < mut_rate: swap_mutation(ch2, rng)
            nxt_pop.append(ch1)
            if len(nxt_pop) < pop_size: nxt_pop.append(ch2)
        pop = np.array(nxt_pop)
        fit = np.array([makespan_of(c) for c in pop])
        if verbose and (g % 20 == 0 or g == n_gen - 1):
            print(f'  gen {g:3d}  best={best_hist[-1]}  pop-best={int(fit.min())}  mean={fit.mean():.0f}')
    order = np.argsort(fit)
    if int(fit[order[0]]) < best_fit:
        best_fit, best_chrom = int(fit[order[0]]), pop[order[0]].copy()
    return {'best': best_fit, 'chrom': best_chrom, 'best_hist': best_hist, 'mean_hist': mean_hist}

# quick smoke test (small budget) to validate the loop
smoke = run_ga(pop_size=20, n_gen=10, seed=0)
print('Smoke test best:', smoke['best'])

## 4. Experiments

### 4.1 Main run — convergence behaviour

Default configuration: `pop=80, generations=150, cx=0.85, mut=0.25, tournament=3, elite=2`. We track best-so-far and population mean.

In [ ]:
t0 = time.time()
main = run_ga(pop_size=80, n_gen=150, cx_rate=0.85, mut_rate=0.25, tour_size=3, n_elite=2, seed=1, verbose=True)
print(f'Main run: best makespan = {main["best"]}  (elapsed {time.time()-t0:.1f}s)')

plt.figure(figsize=(8, 4.2))
plt.plot(main['best_hist'], label='best so far', linewidth=2)
plt.plot(main['mean_hist'], label='population mean', alpha=0.7)
plt.xlabel('Generation'); plt.ylabel('Makespan'); plt.title('GA convergence (pop=80, cx=0.85, mut=0.25, tour=3)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 4.2 Parameter sensitivity

Each configuration is run with a reduced budget (`60` generations) and **3 seeds** to separate signal from luck. We vary one parameter at a time around the default (`pop=80, cx=0.85, mut=0.25, tour=3`).

In [ ]:
def sweep(param_name, values, seeds=(1, 2, 3), base=None, n_gen=60):
    base = base or dict(pop_size=80, cx_rate=0.85, mut_rate=0.25, tour_size=3)
    rows = []
    hists = {}
    for v in values:
        cfg = dict(base); cfg[param_name] = v
        key = {'pop_size': 'pop_size', 'cx_rate': 'cx_rate', 'mut_rate': 'mut_rate', 'tour_size': 'tour_size'}[param_name]
        bests, hs = [], []
        for s in seeds:
            r = run_ga(n_gen=n_gen, seed=s, **cfg)
            bests.append(r['best']); hs.append(r['best_hist'])
            rows.append({'param': param_name, 'value': v, 'seed': s, 'best': r['best']})
        hists[v] = np.mean(hs, axis=0)
        print(f'{param_name}={v}: bests={bests} -> mean {np.mean(bests):.1f} ± {np.std(bests):.1f}')
    return rows, hists

import pandas as pd
all_rows = []

print('--- A: population size ---')
rows_pop, hist_pop = sweep('pop_size', [30, 80, 150])
all_rows += rows_pop

print('--- B: mutation rate ---')
rows_mut, hist_mut = sweep('mut_rate', [0.05, 0.25, 0.6])
all_rows += rows_mut

print('--- C: crossover rate ---')
rows_cx, hist_cx = sweep('cx_rate', [0.5, 0.85, 1.0])
all_rows += rows_cx

print('--- D: tournament size ---')
rows_tour, hist_tour = sweep('tour_size', [2, 3, 5])
all_rows += rows_tour

res = pd.DataFrame(all_rows)
summary = res.groupby(['param', 'value'])['best'].agg(['mean', 'min', 'std']).round(1)
print(summary)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
for a, h, t in [(ax[0,0], hist_pop, 'A: population size'), (ax[0,1], hist_mut, 'B: mutation rate'),
                (ax[1,0], hist_cx, 'C: crossover rate'), (ax[1,1], hist_tour, 'D: tournament size')]:
    for v, hmean in h.items():
        a.plot(hmean, label=f'{v}')
    a.set_xlabel('Generation (60-gen budget, mean of 3 seeds)'); a.set_ylabel('Best-so-far makespan')
    a.set_title(t); a.legend(title='value'); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# bar chart of final means
piv = res.groupby(['param','value'])['best'].mean().unstack(0)
print(piv)

### 4.3 Best schedule — Gantt chart and utilisation

We decode the best chromosome of the main run and visualise it per machine.

In [ ]:
best_chrom = main['chrom']
rec, best_ms = decode_schedule(best_chrom)
print(f'Best makespan: {best_ms}  (lower bound was {lb}, gap {(best_ms-lb)/lb*100:.1f}%)')

# utilisation per machine
busy = np.zeros(N_MACH)
for (_, _, m, s, e) in rec: busy[m] += e - s
util = busy / best_ms * 100
for m in range(N_MACH):
    print(f'Machine {m+1:2d}: busy={int(busy[m]):5d}  utilisation={util[m]:5.1f}%')

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab20(np.linspace(0, 1, N_JOBS))
for (j, o, m, s, e) in rec:
    ax.barh(m, e - s, left=s, height=0.8, color=colors[j % N_JOBS], edgecolor='none')
ax.set_yticks(range(N_MACH)); ax.set_yticklabels([f'M{m+1}' for m in range(N_MACH)])
ax.set_xlabel('Time'); ax.set_title(f'Gantt chart of best schedule (makespan={best_ms}, 550 operations, colours=jobs)')
ax.set_xlim(0, best_ms * 1.02); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([f'M{m+1}' for m in range(N_MACH)], util, color='darkorange')
ax.set_ylabel('Utilisation %'); ax.set_title('Machine utilisation of best schedule')
plt.tight_layout(); plt.show()

## 5. Analysis and conclusions

Observed results on this dataset (50 jobs x 11 ops, 10 machines, lower bound 2073):

- **GA clearly beats baselines.** FIFO job-ordered schedule gives makespan **11504** (all operations of one job block machines sequentially - pathological). Average random permutation gives **~2856** (min 2569 over 30 trials). The GA main run (`pop=80, gen=150, cx=0.85, mut=0.25, tour=3`) reaches **~2119**, i.e. only **2.2% above the theoretical lower bound 2073**. The best configuration in the sweep (`pop=150`) even reached **2089** (<1% gap). The problem is tightly constrained by bottleneck machine 7 (load exactly 2073), so near-optimal means keeping machine 7 almost continuously busy - the utilisation chart confirms **~97.8%** utilisation there.
- **Convergence:** steep improvement in the first ~20-30 generations (from ~2530 to ~2200), then slow fine-tuning down to ~2120. Population mean stays above best-so-far, i.e. diversity is preserved; elitism guarantees monotonic best-so-far.
- **Population size (A):** 30 -> mean 2233, 80 -> 2155, 150 -> 2106 (60-gen budget, 3 seeds). Larger populations win because diversity matters for 550-gene chromosomes; 150 costs proportionally more evaluations with diminishing returns, so 80 is a good cost/quality trade-off.
- **Mutation (B):** 0.05 -> 2146, 0.25 -> 2155, 0.6 -> 2127. All in the same ballpark; very low mutation risks premature convergence, very high mutation adds variance (std ~32-39 vs ~12) but can still help on a short budget by escaping local optima. ~0.2-0.3 (one swap per offspring) is a safe default.
- **Crossover (C):** 0.5 -> 2205, 0.85 -> 2155, 1.0 -> 2135 (std only ~1). JOX recombination is the main driver - high rate (0.85-1.0) is clearly better; 0.5 is visibly worse.
- **Tournament (D):** 2 -> 2159, 3 -> 2155, 5 -> 2161 but with large variance (std ~57, one run stuck at 2240). Size 2 is slightly slow, 5 is too greedy/unstable; 3 balances exploration/exploitation.
- **Recommended setting:** `pop=80, generations=100-150, cx=0.85-1.0, mut=0.25, tour=3, elite=2`, run several seeds and keep the best. Runtime is seconds on a laptop (~0.4 ms per makespan evaluation; 80x150 run ~7.5 s).
- **Limits / next steps:** the list-scheduler decoder only yields semi-active schedules and single-swap mutation is a weak local move; adding local search (e.g. critical-path hill climbing / tabu) or adaptive mutation would likely close the last 1-2% to the lower bound. Results vary with seed - always report mean +- std over repeats.



Implement the Particle Swarm Optimisation algorithm to find the minimum of the following function:

`f(x,y) = (1.5 – x -xy)^2 + (2.25 – x + xy^2)^2 + (2.625 – x + xy^3)^2 for x,y in range <-4.5, 4.5>`

Test your algorithm using different parameters and analyse the results.

In [ ]:
def target_function(x,y):
    return (1.5 - x -(x*y))**2 + (2.25 - x + x*y**2)**2 + (2.625 - x + xy**3)**2